[<img src="imagens/colab-badge.png" style="width:16%; vertical-align:middle;">](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.fr/cap04/cap04.EPs_aluno.ipynb)
[<img src="imagens/github-badge.png" style="width:19%; vertical-align:middle;">](https://github.com/fzampirolli/pdi-vc)

## 💻 **Partie Pratique avec Exercices de Programmation**


Cette liste transforme les concepts du Chapitre 4 en un parcours pratique de segmentation et de morphologie mathématique. Les exercices de programmation commencent par le seuillage et progressent jusqu'à l'étiquetage et les descripteurs de composants, toujours avec des matrices de petite taille afin que chaque pixel puisse être vérifié à la main.

> ### ❗ Règle commune des exercices de programmation morphologiques
>
> Dans les opérations avec voisinage, **ne faites pas de padding**. Pour chaque pixel, évaluez uniquement les positions de l'élément structurant qui se trouvent dans le domaine de l'image. C'est la même idée que les implémentations pédagogiques dans `morph.py`, comme `mm.dil0`, `mm.ero0`, `mm.dil1` et `mm.label0` : le voisinage est découpé par le domaine valide de l'image.

### 🎯 Objectif de ce carnet

Ce carnet permet de développer, valider, organiser et tester des solutions d' **Exercices de Programmation (EP)** dans des environnements interactifs, tels que Colab, avec les mêmes cas de test que Moodle, en y copiant uniquement au moment d'enregistrer la note officielle.

#### *Téléchargement*

Téléchargez `morph.py` et `testsuite.py` en exécutant la cellule ci-dessous :

In [ ]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup(testsuite=True)
from morph import mm
from testsuite import TestSuite

#### Exécution des tests
Pour évaluer les tests, exécutez `TestSuite("EP04_01.extensão").run()` dans une nouvelle cellule, en remplaçant l’extension par celle du langage utilisé (`.py`, `.java`, `.c`, `.cpp`, `.js` ou `.r`). Le système télécharge les cas de test depuis GitHub, exécute le programme et calcule automatiquement la note.

Pour tester directement du code Python, sans enregistrer de fichier, utilisez `run_code(codigo)` en passant le code comme *chaîne de caractères* dans une variable `codigo` :

```python
codigo = """
from morph import mm
# ... votre code ici ...
"""
TestSuite("EP04_01").run_code(codigo)
```

### EP04_01 🎚️ Seuillage global par seuil fixe

Dans les **scanners de documents** et les **systèmes de lecture de codes-barres**, la première étape du traitement consiste toujours à séparer ce qui est « objet » (encre, texte, barres) de ce qui est « fond » (papier, emballage). Le **seuillage global** fait exactement cela : il compare chaque pixel à un seuil unique $T$ et décide, en temps réel, s'il appartient à la classe claire ou à la classe sombre. C'est l'opérateur de segmentation le plus simple — et pourtant, il est à l'origine d'une grande partie des *pipelines* industriels d'inspection visuelle.
Voir la simulation de cet EP dans [Figura 4.1](#fig-04-sim-ep0401-limiar).

#### 📋 Directives d'implémentation

1. **Dimensions :** Lire les entiers $L$ (lignes) et $C$ (colonnes).
2. **Seuil :** Lire l'entier $T$ (seuil de décision).
3. **Données :** Lire les valeurs entières de la matrice originale ligne par ligne.
4. **Mappage :** Pour chaque pixel $p$, calculer la nouvelle valeur à l'aide de l'équation :

$$
p' =
\begin{cases}
255, & \text{si } p > T \\
0, & \text{si } p \le T
\end{cases}
$$
5. **Sortie :** Afficher la matrice binarisée avec les dimensions $L \times C$.

#### 📌 Contraintes computationnelles

* **Binarisation :** La sortie contient **uniquement** les valeurs $0$ ou $255$.
* **Comparaison stricte :** Le critère utilise $> T$ (les pixels égaux à $T$ deviennent du fond).
* **Type :** Le résultat final doit être entier.
* **Remarque :** Cet EP suit la convention d'OpenCV (`cv2.THRESL_BINARY`) : seuls les pixels avec une valeur **supérieure à** $T$ deviennent blancs (`255`) ; les pixels avec une valeur **égale à** $T$ restent noirs (`0`).

#### 🧠 Fondement théorique

| Paramètre | Type | Impact visuel |
|-----------|------|---------------|
| **$T$ petit** | Entier | La plupart des pixels deviennent blancs |
| **$T$ grand** | Entier | La plupart des pixels deviennent noirs |
| **$T$ bien choisi** | Entier | Sépare nettement l'objet et le fond |

#### 📦 Spécification d'entrée et de sortie (VPL)

**Entrée :**

* Ligne 1 : Entier $L$.
* Ligne 2 : Entier $C$.
* Ligne 3 : Entier $T$.
* Lignes suivantes : Éléments entiers de la matrice originale.

**Sortie :**

* Matrice binarisée en $L$ lignes et $C$ colonnes, valeurs $0$ ou $255$ séparées par des espaces.

#### 📌 Exemples

| Entrée | Sortie | Remarque |
|--------|--------|----------|
| 2<br>4<br>100<br>0 99 100 180<br>255 30 120 80 | 0 0 0 255<br>255 0 255 0 | $T=100$ : seuls les pixels avec une valeur supérieure à 100 deviennent blancs ; <br>par conséquent, 99 et 100 deviennent noirs. |
| 1<br>3<br>0<br>0 50 255 | 0 255 255 | $T=0$ : seuls les pixels avec une valeur strictement supérieure à 0 deviennent blancs. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0401-limiar" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🎚️ Simulateur EP04_01 : Seuillage global</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">p' = (p > T) ? 255 : 0</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">👆 Cliquez sur une cellule de la <b>Entrée originale</b> pour assombrir le pixel (−30) et cliquez avec le bouton droit pour éclaircir (+30). Ajustez le seuil T pour la binarisation.</p>

    <!-- Controle do Limiar T -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#2980b9;">T (Seuil)</label>
        <span id="sim_ep0401_vl_t" style="font-family:monospace;font-size:12px;font-weight:700;color:#2980b9;">128</span>
      </div>
      <input type="range" id="sim_ep0401_sl_t" min="0" max="255" step="1" value="128" style="width:100%;cursor:pointer;">
    </div>

    <!-- Comparativo Lado a Lado: Entrada vs Resultado Binarizado -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Entrada Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Entrée originale (cliquable)</span>
        <div id="sim_ep0401_grid_orig" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0401_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nouvelle image</button>
      </div>

      <!-- Resultado Binarizado -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Résultat binarisé (p')</span>
        <div id="sim_ep0401_grid_new" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0401_btnReset" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">↩ Réinitialiser le seuil (T = 128)</button>
      </div>

    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0401_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      Formule appliquée : <b>(p > 128) ? 255 : 0</b>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0401(root){
    if (!root || root.dataset.simEp0401Init) return;
    root.dataset.simEp0401Init = "1";

    var slT      = root.querySelector('#sim_ep0401_sl_t');
    var vlT      = root.querySelector('#sim_ep0401_vl_t');
    var gridOrig = root.querySelector('#sim_ep0401_grid_orig');
    var gridNew  = root.querySelector('#sim_ep0401_grid_new');
    var debugDiv = root.querySelector('#sim_ep0401_debug');

    var btnNew   = root.querySelector('#sim_ep0401_btnNew');
    var btnReset = root.querySelector('#sim_ep0401_btnReset');

    var pixels = Array(16).fill(0).map(function(){ return Math.floor(Math.random() * 256); });

    function renderOrig() {
      gridOrig.innerHTML = '';
      pixels.forEach(function(p, idx) {
        var cellO = document.createElement('div');
        var fgColorO = p > 128 ? '#000000' : '#ffffff';
        cellO.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;cursor:pointer;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgColorO + ';box-sizing:border-box;';
        cellO.textContent = p;
        cellO.title = 'Clique esquerdo: escurece (-30) | Botão direito: clareia (+30)';

        cellO.addEventListener('click', function(e) {
          e.preventDefault();
          pixels[idx] = Math.max(0, pixels[idx] - 30);
          render();
        });

        cellO.addEventListener('contextmenu', function(e) {
          e.preventDefault();
          pixels[idx] = Math.min(255, pixels[idx] + 30);
          render();
        });

        gridOrig.appendChild(cellO);
      });
    }

    function render() {
      var T = parseInt(slT.value, 10) || 0;
      vlT.textContent = T;
      debugDiv.innerHTML = 'Fórmula aplicada: <b>(p > ' + T + ') ? 255 : 0</b>';

      renderOrig();
      gridNew.innerHTML = '';

      pixels.forEach(function(p) {
        var res = (p > T) ? 255 : 0;
        var fgColorN = res > 128 ? '#000000' : '#ffffff';
        var cellN = document.createElement('div');
        cellN.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;background:rgb(' + res + ',' + res + ',' + res + ');color:' + fgColorN + ';box-sizing:border-box;';
        cellN.textContent = res;
        gridNew.appendChild(cellN);
      });
    }

    slT.addEventListener('input', render);

    btnNew.addEventListener('click', function() {
      pixels = Array(16).fill(0).map(function(){ return Math.floor(Math.random() * 256); });
      render();
    });

    btnReset.addEventListener('click', function() {
      slT.value = '128';
      render();
    });

    render();
  }

  function tryInitSimEP0401(){
    var root = document.getElementById('sim-ep0401-limiar');
    if (root) initSimEP0401(root); else setTimeout(tryInitSimEP0401, 200);
  }
  tryInitSimEP0401();
})();
</script>
</div>
""")

**Figura 4.1:** Simulateur EP04_01 : Seuillage global par seuil fixe (p


<figure id="fig-04-sim-ep0401-limiar">
  <img src="imagens/fig-04-sim-ep0401-limiar.png" alt=" Simulateur EP04_01 : Seuillage global par seuil fixe (p' = (p > T) ? 255 : 0) " style="max-width:80%" />
  <figcaption><strong>Figura 4.1:</strong>  Simulateur EP04_01 : Seuillage global par seuil fixe (p' = (p > T) ? 255 : 0) </figcaption>
</figure>

In [ ]:
%%writefile EP04_01.py
# Code Python

In [ ]:
TestSuite("EP04_01.py").run()

### EP04_02 📊 Seuillage automatique d’Otsu

Choisir manuellement le seuil $T$ fonctionne lorsque l’éclairage est stable, mais en **microscopie numérique** et en **inspection de lames de sang**, chaque échantillon présente un contraste différent — un seuil fixe échouerait d’une image à l’autre. La **méthode d’Otsu** résout ce problème en trouvant, de manière autonome, le seuil qui **maximise la séparation statistique** entre les deux classes de pixels, rendant la segmentation automatique et adaptative.
Voir dans [Figura 4.2](#fig-04-sim-ep0402-otsu) une simulation de cet EP.

#### 📋 Directives d’implémentation

1. **Dimensions :** Lire les entiers $L$ (lignes) et $C$ (colonnes).
2. **Données :** Lire les valeurs entières de la matrice originale ligne par ligne.
3. **Histogramme :** Construire l’histogramme $h[i]$, $i=0,\dots,255$, en comptant combien de pixels ont la valeur $i$.
4. **Recherche du seuil :** Pour chaque candidat $T$ de $1$ à $255$, calculer la **variance inter-classes** :
$$
\sigma_B^2(T) = \frac{n_0 \cdot n_1}{N^2}\,(m_0 - m_1)^2
$$
où $n_0,n_1$ sont les quantités de pixels ayant une valeur $<T$ et $\geq T$, $m_0,m_1$ sont leurs moyennes, et $N=L\times C$.

5. **Choix :** Le seuil optimal $T^*$ est celui qui maximise $\sigma_B^2(T)$ (en cas d’égalité, conserver le **premier** trouvé).
6. **Application :** Binariser l’image en utilisant $T^*$, en appliquant :
$$
p' =
\begin{cases}
255, & \text{si } p > T^* \\
0, & \text{si } p \le T^*
\end{cases}
$$

#### 📌 Contraintes computationnelles

* **Candidats valides :** Ignorer $T$ qui laisse $n_0=0$ ou $n_1=0$ (classe vide).
* **Égalité :** Toujours conserver le **premier** $T$ qui a atteint la valeur maximale de $\sigma_B^2$.
* **Type :** $T^*$ et la matrice de sortie doivent être des entiers.
* **Convention OpenCV :** La binarisation suit `cv2.THRESL_BINARY` ; les pixels ayant une valeur exactement égale à $T^*$ deviennent noirs.

#### 🧠 Fondements théoriques

| Concept | Signification | Impact |
|----------|-------------|---------|
| **$\sigma_B^2(T)$ élevée** | Classes bien séparées en $T$ | $T$ est un bon candidat comme seuil |
| **Histogramme bimodal** | Deux « pics » distincts | Otsu trouve le creux entre eux |
| **Histogramme unimodal** | Un seul « pic » | Otsu choisit toujours *un* $T$, mais la segmentation est peu fiable |

#### 📦 Spécification d’entrée et de sortie (VPL)

**Entrée :**

* Ligne 1 : Entier $L$.
* Ligne 2 : Entier $C$.
* Lignes suivantes : Éléments entiers de la matrice originale.

**Sortie :**

* Matrice binarisée en $L$ lignes et $C$ colonnes, valeurs $0$ ou $255$.

#### 📌 Exemples

| Entrée | Sortie | Observation |
|---------|-------|------------|
| 4<br>4<br>12 12 12 200<br>12 12 200 200<br>12 200 200 200<br>200 200 200 200 | 0 0 0 255<br>0 0 255 255<br>0 255 255 255<br>255 255 255 255 | Histogramme bimodal net : 12 et 200 |
| 1<br>2<br>10 250 | 0 250 | Deux valeurs seulement : $T^*$ reste sur la plus grande |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0402-otsu" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">📊 Simulateur EP04_02 : Otsu Automatique</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">T* = argmax σ²_B(T)</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">👆 Clic gauche assombrit (−25) et clic droit éclaircit (+25) les pixels d'entrée. Observez le seuil optimal T* s'ajuster dynamiquement à l'histogramme.</p>

    <!-- Painel do Histograma e T* -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:14px;margin-bottom:16px;">
      <div id="sim_ep0402_hist" style="display:flex;align-items:flex-end;gap:2px;height:100px;margin-bottom:8px;border-bottom:1px solid #e4dcc8;padding-bottom:2px;"></div>
      <p id="sim_ep0402_info" style="text-align:center;font-size:11.5px;font-family:monospace;font-weight:700;color:#26241d;margin:0;">T* = −</p>
    </div>

    <!-- Comparativo Lado a Lado: Entrada Clicável vs Resultado Otsu -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Entrada Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Entrée Originale (Cliquable)</span>
        <div id="sim_ep0402_grid_orig" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Resultado Otsu -->
      <div style="background:#fafaf7;border:1px solid #16a085;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#16a085;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Résultat Otsu (p')</span>
        <div id="sim_ep0402_grid_new" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Botão de Nova Imagem -->
    <div style="text-align:center;">
      <button id="sim_ep0402_btnNew" style="padding:6px 14px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nouvelle Image (Deux Groupes)</button>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0402(root){
    if (!root || root.dataset.simEp0402Init) return;
    root.dataset.simEp0402Init = "1";

    var gridOrig = root.querySelector('#sim_ep0402_grid_orig');
    var gridNew  = root.querySelector('#sim_ep0402_grid_new');
    var info     = root.querySelector('#sim_ep0402_info');
    var hist     = root.querySelector('#sim_ep0402_hist');
    var btnNew   = root.querySelector('#sim_ep0402_btnNew');

    var pixels = [];

    function generate() {
      var c1 = 20 + Math.floor(Math.random() * 40);
      var c2 = 180 + Math.floor(Math.random() * 60);
      pixels = [];
      for (var i = 0; i < 16; i++) {
        var base = (Math.random() < 0.5) ? c1 : c2;
        pixels.push(Math.max(0, Math.min(255, base + Math.floor(Math.random() * 16 - 8))));
      }
    }

    function otsu(pix) {
      var histArr = new Array(256).fill(0);
      pix.forEach(function(p){ histArr[p]++; });
      var N = pix.length, bestVar = -1, bestT = 0;
      var total = pix.reduce(function(a, b){ return a + b; }, 0);

      for (var T = 1; T < 256; T++) {
        var n0 = 0, s0 = 0;
        for (var i = 0; i < T; i++) {
          n0 += histArr[i];
          s0 += i * histArr[i];
        }
        var n1 = N - n0, s1 = total - s0;
        if (n0 === 0 || n1 === 0) continue;
        var m0 = s0 / n0, m1 = s1 / n1;
        var v = (n0 * n1) * (m0 - m1) * (m0 - m1) / (N * N);
        if (v > bestVar) {
          bestVar = v;
          bestT = T;
        }
      }
      return bestT;
    }

    function renderOrig() {
      gridOrig.innerHTML = '';
      pixels.forEach(function(p, idx) {
        var cellO = document.createElement('div');
        var fgColorO = p > 128 ? '#000000' : '#ffffff';
        cellO.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;cursor:pointer;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgColorO + ';box-sizing:border-box;';
        cellO.textContent = p;
        cellO.title = 'Clique esquerdo: escurece (-25) | Botão direito: clareia (+25)';

        cellO.addEventListener('click', function(e) {
          e.preventDefault();
          pixels[idx] = Math.max(0, pixels[idx] - 25);
          render();
        });

        cellO.addEventListener('contextmenu', function(e) {
          e.preventDefault();
          pixels[idx] = Math.min(255, pixels[idx] + 25);
          render();
        });

        gridOrig.appendChild(cellO);
      });
    }

    function renderHist(T) {
      hist.innerHTML = '';
      var histArr = new Array(256).fill(0);
      pixels.forEach(function(p){ histArr[p]++; });
      var maxH = Math.max.apply(null, histArr);

      for (var i = 0; i < 256; i += 4) {
        var h = (histArr[i] / (maxH || 1)) * 100;
        var bar = document.createElement('div');
        var col = (i >= T) ? '#2980b9' : '#8a8371';
        bar.style.cssText = 'flex:1;height:' + h + '%;background:' + col + ';border-radius:2px 2px 0 0;';
        hist.appendChild(bar);
      }
    }

    function render() {
      var T = otsu(pixels);
      info.innerHTML = 'T* encontrado = <b>' + T + '</b>';
      renderOrig();
      renderHist(T);

      gridNew.innerHTML = '';
      pixels.forEach(function(p) {
        var res = (p > T) ? 255 : 0;
        var fgColorN = res > 128 ? '#000000' : '#ffffff';
        var cellN = document.createElement('div');
        cellN.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;background:rgb(' + res + ',' + res + ',' + res + ');color:' + fgColorN + ';box-sizing:border-box;';
        cellN.textContent = res;
        gridNew.appendChild(cellN);
      });
    }

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0402(){
    var root = document.getElementById('sim-ep0402-otsu');
    if (root) initSimEP0402(root); else setTimeout(tryInitSimEP0402, 200);
  }
  tryInitSimEP0402();
})();
</script>
</div>
""")

**Figura 4.2:** Simulateur EP04_02 : Seuillage automatique d


<figure id="fig-04-sim-ep0402-otsu">
  <img src="imagens/fig-04-sim-ep0402-otsu.png" alt=" Simulateur EP04_02 : Seuillage automatique d'Otsu (T* = argmax σ²_B(T)) " style="max-width:80%" />
  <figcaption><strong>Figura 4.2:</strong>  Simulateur EP04_02 : Seuillage automatique d'Otsu (T* = argmax σ²_B(T)) </figcaption>
</figure>

In [ ]:
%%writefile EP04_02.py
# Code Python

In [ ]:
TestSuite("EP04_02.py").run()

### EP04_03 🌱 Dilatation binaire plane (mm.dil0)

En **microscopie de particules** et en **OCR de plaques d'immatriculation usées**, les traits fins ou discontinus doivent être « épaissis » pour que la reconnaissance fonctionne. La **dilatation morphologique** fait exactement cela : elle étend les régions claires à l'aide d'un élément structurant $B$ — la même opération implémentée dans `morph.py` comme `mm.dil0(f, B)`, utilisée lorsque $B$ est **plat** (sans poids, uniquement $0$/$1$).
Voir dans [Figura 4.3](#fig-04-sim-ep0403-dilatacao) une simulation de cet EP.

#### 📋 Directives d'implémentation

1. **Dimensions de l'image :** Lire les entiers $L$ (lignes) et $C$ (colonnes) de $f$.
2. **Dimensions de $B$ :** Lire les entiers $L_B$ (lignes) et $C_B$ (colonnes) de l'élément structurant.
3. **Élément structurant :** Lire la matrice $B$ avec des valeurs $0$ ou $1$, ligne par ligne.
4. **Données :** Lire la matrice $f$ (l'image originale), ligne par ligne.
5. **Réflexion :** Construire $B_{ref}$, la version de $B$ réfléchie à $180°$ (lignes et colonnes inversées) — exactement comme le fait `mm.dil0` en interne.
6. **Voisinage sans padding :** Pour chaque pixel $(y,x)$, parcourir les positions $(by,bx)$ de $B_{ref}$ centrées en $(y,x)$, en utilisant le décalage
$$
v_y = y + by + o_y,\quad v_x = x + bx + o_x,\quad o_y=-\tfrac{L_B}{2}+0{,}5,\quad o_x=-\tfrac{C_B}{2}+0{,}5
$$
**Écarter** tout $(v_y,v_x)$ hors de $[0,L)\times[0,C)$ — **ne pas remplir avec des zéros**.
7. **Mappage :** Calculer chaque pixel de sortie comme le **maximum** entre $f(y,x)$ et tous les $f(v_y,v_x)$ valides dont la position correspondante dans $B_{ref}$ vaut $1$ :
$$
g(y,x) = \max\Big(f(y,x),\ \max_{\substack{(v_y,v_x)\ \text{valide}\\ B_{ref}(by,bx)=1}} f(v_y,v_x)\Big)
$$
8. **Sortie :** Afficher la matrice $g$ avec les dimensions $L \times C$.

#### 📌 Contraintes computationnelles

* **Sans padding :** Ne jamais inventer de voisins hors de l'image ; n'utiliser que ceux qui existent réellement.
* **Réflexion obligatoire :** $B$ doit être réfléchi avant d'être appliqué (c'est ce qui distingue `mm.dil0` d'une simple recherche de maximum).
* **Robustesse des bords :** Si aucune position valide de $B_{ref}=1$ ne tombe dans le domaine pour un pixel donné, celui-ci **conserve sa valeur originale**.

#### 🧠 Fondement théorique

| Concept | Signification | Impact visuel |
|----------|-------------|-----------------|
| **Dilatation** | $g \geq f$ toujours (extensive) | Les régions claires croissent, les trous sombres rétrécissent |
| **$B$ plus grand** | Voisinage plus large | Croissance plus agressive |
| **Réflexion de $B$** | $B_{ref}(y,x) = B(-y,-x)$ | Garantit la définition formelle de Minkowski de la dilatation |

#### 📦 Spécification d'entrée et de sortie (VPL)

**Entrée :**

* Ligne 1 : entier $L$.
* Ligne 2 : entier $C$.
* Ligne 3 : entier $L_B$.
* Ligne 4 : entier $C_B$.
* Les $L_B$ lignes suivantes : éléments entiers ($0$ ou $1$) de la matrice $B$.
* Les $L$ lignes suivantes : éléments entiers de la matrice $f$.

**Sortie :**

* Matrice $g$ en $L$ lignes et $C$ colonnes, valeurs entières séparées par des espaces.

#### 📌 Exemples

| Entrée | Sortie | Observation |
|---------|-------|------------|
| 3<br>3<br>3<br>3<br>0 1 0<br>1 1 1<br>0 1 0<br>0 0 0<br>0 9 0<br>0 0 0 | 0 9 0<br>9 9 9<br>0 9 0 | $B$ en croix symétrique : point isolé se dilate en croix |
| 1<br>4<br>1<br>3<br>1 1 1<br>10 200 5 80 | 200 200 200 80 | $B$ horizontal : chaque pixel « attire » le maximum des voisins de la ligne |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0403-dilatacao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🌱 Simulateur EP04_03 : Dilatation plane (mm.dil0)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = f ⊕ B</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Alternez l'élément structurant B (ou sélectionnez les préréglages) et cliquez sur les cellules de l'image originale f pour allumer ou éteindre les pixels.</p>

    <!-- Painel do Elemento Estruturante B -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:14px;margin-bottom:16px;text-align:center;">
      <span style="font-size:10px;font-weight:700;color:#16a085;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Élément structurant B (Cliquez pour basculer 0/1)</span>
      <div id="sim_ep0403_grid_B" style="display:grid;grid-template-columns:repeat(3, 38px);gap:4px;justify-content:center;margin-bottom:12px;user-select:none;"></div>
      
      <!-- Presets de B -->
      <div style="display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
        <button id="sim_ep0403_btnCross" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">➕ Croix</button>
        <button id="sim_ep0403_btnBox" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">⬛ Carré</button>
        <button id="sim_ep0403_btnDiag" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">⤫ Diagonale</button>
      </div>
    </div>

    <!-- Comparativo Lado a Lado: Entrada Original f vs Dilatada g -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem Original f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Image originale f (5×5)</span>
        <div id="sim_ep0403_grid_orig" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0403_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nouvelle image</button>
      </div>

      <!-- Imagem Dilatada g -->
      <div style="background:#fafaf7;border:1px solid #16a085;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#16a085;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Dilatée g (f ⊕ B)</span>
        <div id="sim_ep0403_grid_new" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
        <div style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid transparent;background:transparent;color:transparent;user-select:none;">&nbsp;</div>
      </div>

    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0403_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      g(y,x) = max sur les voisins valides de B réfléchi
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0403(root){
    if (!root || root.dataset.simEp0403Init) return;
    root.dataset.simEp0403Init = "1";

    var gB      = root.querySelector('#sim_ep0403_grid_B');
    var gO      = root.querySelector('#sim_ep0403_grid_orig');
    var gN      = root.querySelector('#sim_ep0403_grid_new');
    var debugDiv= root.querySelector('#sim_ep0403_debug');

    var btnNew   = root.querySelector('#sim_ep0403_btnNew');
    var btnCross = root.querySelector('#sim_ep0403_btnCross');
    var btnBox   = root.querySelector('#sim_ep0403_btnBox');
    var btnDiag  = root.querySelector('#sim_ep0403_btnDiag');

    var B = [[0, 1, 0], [1, 1, 1], [0, 1, 0]];
    var L = 5, C = 5, pixels = [];

    function generate() {
      pixels = [];
      for (var y = 0; y < L; y++) {
        var row = [];
        for (var x = 0; x < C; x++) {
          row.push((Math.random() < 0.78) ? 0 : 1);
        }
        pixels.push(row);
      }
    }

    function reflect(M) {
      var n = M.length, out = [];
      for (var i = n - 1; i >= 0; i--) {
        out.push(M[i].slice().reverse());
      }
      return out;
    }

    function dilate(f, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var Bref = reflect(Bm);
      var g = [];
      for (var y = 0; y < L; y++) g.push(f[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bref[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && f[vy][vx] > g[y][x]) {
                g[y][x] = f[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function renderB() {
      gB.innerHTML = '';
      for (var by = 0; by < 3; by++) {
        for (var bx = 0; bx < 3; bx++) {
          (function(byy, bxx){
            var c = document.createElement('div');
            c.style.cssText = 'width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:12px;font-weight:700;font-family:monospace;border-radius:6px;cursor:pointer;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;';
            c.style.background = B[byy][bxx] ? '#16a085' : '#fafaf7';
            c.style.color = B[byy][bxx] ? '#ffffff' : '#8a8371';
            c.textContent = B[byy][bxx];

            c.addEventListener('click', function(){
              B[byy][bxx] = 1 - B[byy][bxx];
              renderB();
              render();
            });
            gB.appendChild(c);
          })(by, bx);
        }
      }
    }

    function render() {
      var g = dilate(pixels, B);
      gO.innerHTML = '';
      gN.innerHTML = '';

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          (function(yy, xx){
            var p = pixels[yy][xx] ? 255 : 30;
            var fgO = p > 128 ? '#000000' : '#ffffff';
            var cO = document.createElement('div');
            cO.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;cursor:pointer;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgO + ';box-sizing:border-box;';
            cO.textContent = pixels[yy][xx];

            cO.addEventListener('click', function(){
              pixels[yy][xx] = 1 - pixels[yy][xx];
              render();
            });
            gO.appendChild(cO);
          })(y, x);

          var r = g[y][x] ? 255 : 30;
          var fgN = r > 128 ? '#000000' : '#ffffff';
          var cN = document.createElement('div');
          cN.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + r + ',' + r + ',' + r + ');color:' + fgN + ';box-sizing:border-box;';
          cN.textContent = g[y][x];
          gN.appendChild(cN);
        }
      }
    }

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    btnCross.addEventListener('click', function(){
      B = [[0, 1, 0], [1, 1, 1], [0, 1, 0]];
      renderB();
      render();
    });

    btnBox.addEventListener('click', function(){
      B = [[1, 1, 1], [1, 1, 1], [1, 1, 1]];
      renderB();
      render();
    });

    btnDiag.addEventListener('click', function(){
      B = [[1, 0, 1], [0, 1, 0], [1, 0, 1]];
      renderB();
      render();
    });

    generate();
    renderB();
    render();
  }

  function tryInitSimEP0403(){
    var root = document.getElementById('sim-ep0403-dilatacao');
    if (root) initSimEP0403(root); else setTimeout(tryInitSimEP0403, 200);
  }
  tryInitSimEP0403();
})();
</script>
</div>
""")

**Figura 4.3:** Simulateur EP04_03 : Dilatation binaire plane (g = f ⊕ B)


<figure id="fig-04-sim-ep0403-dilatacao">
  <img src="imagens/fig-04-sim-ep0403-dilatacao.png" alt=" Simulateur EP04_03 : Dilatation binaire plane (g = f ⊕ B) " style="max-width:80%" />
  <figcaption><strong>Figura 4.3:</strong>  Simulateur EP04_03 : Dilatation binaire plane (g = f ⊕ B) </figcaption>
</figure>

In [ ]:
%%writefile EP04_03.py
# Code Python

In [ ]:
TestSuite("EP04_03.py").run()

### EP04_04 🪨 Érosion binaire plane (mm.ero0)

Si la dilatation épaissit, l'**érosion** affine. Dans les **systèmes de comptage de cellules**, elle est utilisée pour **séparer les cellules qui se touchent** : en « mangeant » les bords de chaque région, les connexions fines entre objets disparaissent avant même qu'un comptage soit effectué. Dans `morph.py`, c'est l'opération `mm.ero0(f, B)` — le **dual** exact de la dilatation, et la seule des deux qui **ne** réfléchit pas l'élément structurant.
Voir dans [Figura 4.4](#fig-04-sim-ep0404-erosao) une simulation de cet EP.

#### 📋 Directives d'implémentation

1. **Dimensions de l'image :** Lire les entiers $L$ (lignes) et $C$ (colonnes) de $f$.
2. **Dimensions de $B$ :** Lire les entiers $L_B$ (lignes) et $C_B$ (colonnes) de l'élément structurant.
3. **Élément structurant :** Lire la matrice $B$ avec des valeurs $0$ ou $1$, ligne par ligne.
4. **Données :** Lire la matrice $f$ (l'image originale), ligne par ligne.
5. **Voisinage sans padding (sans réflexion !) :** Pour chaque pixel $(y,x)$, parcourir les positions $(by,bx)$ de $B$ **dans l'ordre original** (sans réfléchir), en utilisant le même décalage que dans l'EP04_03 :
$$
v_y = y + by + o_y,\quad v_x = x + bx + o_x,\quad o_y=-\tfrac{L_B}{2}+0{,}5,\quad o_x=-\tfrac{C_B}{2}+0{,}5
$$

**Écarter** tout $(v_y,v_x)$ hors de $[0,L)\times[0,C)$.
6. **Mappage :** Calculer chaque pixel de sortie comme le **minimum** entre $f(y,x)$ et tous les $f(v_y,v_x)$ valides dont la position correspondante dans $B$ vaut $1$ :
$$
g(y,x) = \min\Big(f(y,x),\ \min_{\substack{(v_y,v_x)\ \text{valide}\\ B(by,bx)=1}} f(v_y,v_x)\Big)
$$
7. **Sortie :** Afficher la matrice $g$ avec des dimensions $L \times C$.

#### 📌 Contraintes de calcul

* **Sans réflexion :** Contrairement à la dilatation, $B$ est utilisé **exactement comme lu** — réfléchir ici serait une erreur conceptuelle grave.
* **Sans padding :** Les voisins hors de l'image sont simplement ignorés, jamais traités comme $0$.
* **Robustesse de bord :** Si aucune position valide de $B=1$ ne tombe dans le domaine, le pixel conserve sa valeur originale.

#### 🧠 Fondements théoriques

| Concept | Signification | Impact visuel |
|----------|-------------|-----------------|
| **Érosion** | $g \leq f$ toujours (anti-extensive) | Les régions claires rétrécissent, le bruit ponctuel disparaît |
| **Dualité** | $\text{ero}(f,B) = -\text{dil}(-f, B_{ref})$ | Érosion et dilatation sont des « miroirs » mathématiques |
| **$B$ plus grand** | Érosion plus agressive | Les objets fins disparaissent complètement |

#### 📦 Spécification d'entrée et de sortie (VPL)

**Entrée :**

* Ligne 1 : Entier $L$.
* Ligne 2 : Entier $C$.
* Ligne 3 : Entier $L_B$.
* Ligne 4 : Entier $C_B$.
* $L_B$ lignes suivantes : éléments entiers ($0$ ou $1$) de la matrice $B$.
* $L$ lignes suivantes : éléments entiers de la matrice $f$.

**Sortie :**

* Matrice $g$ en $L$ lignes et $C$ colonnes, valeurs entières séparées par des espaces.

#### 📌 Exemples

| Entrée | Sortie | Observation |
|---------|-------|------------|
| 3<br>3<br>3<br>3<br>0 1 0<br>1 1 1<br>0 1 0<br>9 9 9<br>9 0 9<br>9 9 9 | 9 0 9<br>0 0 0<br>9 0 9 | Le « trou » central (0) se propage en croix |
| 1<br>4<br>1<br>3<br>1 1 1<br>10 200 5 80 | 10 5 5 80 | $B$ horizontal : chaque pixel « tire » le minimum des voisins de la ligne |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0404-erosao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🪨 Simulateur EP04_04 : Érosion plane (mm.ero0)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = f ⊖ B</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Alternez l'élément structurant B (ou sélectionnez les préréglages) et cliquez sur les cellules de l'image originale f pour allumer ou éteindre des pixels.</p>

    <!-- Painel do Elemento Estruturante B -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:14px;margin-bottom:16px;text-align:center;">
      <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Élément structurant B (Cliquer pour alterner 0/1)</span>
      <div id="sim_ep0404_grid_B" style="display:grid;grid-template-columns:repeat(3, 38px);gap:4px;justify-content:center;margin-bottom:12px;user-select:none;"></div>
      
      <!-- Presets de B -->
      <div style="display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
        <button id="sim_ep0404_btnCross" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">➕ Croix</button>
        <button id="sim_ep0404_btnBox" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">⬛ Carré</button>
        <button id="sim_ep0404_btnDiag" style="padding:5px 10px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">⤫ Diagonale</button>
      </div>
    </div>

    <!-- Comparativo Lado a Lado: Entrada Original f vs Erodida g -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem Original f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Image originale f (5×5)</span>
        <div id="sim_ep0404_grid_orig" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0404_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nouvelle image</button>
      </div>

      <!-- Imagem Erodida g -->
      <div style="background:#fafaf7;border:1px solid #c0392b;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Érodée g (f ⊖ B)</span>
        <div id="sim_ep0404_grid_new" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
        <div style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid transparent;background:transparent;color:transparent;user-select:none;">&nbsp;</div>
      </div>

    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0404_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      g(y,x) = min sur les voisins valides de B (sans réflexion)
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0404(root){
    if (!root || root.dataset.simEp0404Init) return;
    root.dataset.simEp0404Init = "1";

    var gB      = root.querySelector('#sim_ep0404_grid_B');
    var gO      = root.querySelector('#sim_ep0404_grid_orig');
    var gN      = root.querySelector('#sim_ep0404_grid_new');
    var debugDiv= root.querySelector('#sim_ep0404_debug');

    var btnNew   = root.querySelector('#sim_ep0404_btnNew');
    var btnCross = root.querySelector('#sim_ep0404_btnCross');
    var btnBox   = root.querySelector('#sim_ep0404_btnBox');
    var btnDiag  = root.querySelector('#sim_ep0404_btnDiag');

    var B = [[0, 1, 0], [1, 1, 1], [0, 1, 0]];
    var L = 5, C = 5, pixels = [];

    function generate() {
      pixels = [];
      for (var y = 0; y < L; y++) {
        var row = [];
        for (var x = 0; x < C; x++) {
          row.push((Math.random() < 0.78) ? 1 : 0);
        }
        pixels.push(row);
      }
    }

    function erode(f, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = [];
      for (var y = 0; y < L; y++) g.push(f[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bm[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && f[vy][vx] < g[y][x]) {
                g[y][x] = f[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function renderB() {
      gB.innerHTML = '';
      for (var by = 0; by < 3; by++) {
        for (var bx = 0; bx < 3; bx++) {
          (function(byy, bxx){
            var c = document.createElement('div');
            c.style.cssText = 'width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:12px;font-weight:700;font-family:monospace;border-radius:6px;cursor:pointer;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;';
            c.style.background = B[byy][bxx] ? '#c0392b' : '#fafaf7';
            c.style.color = B[byy][bxx] ? '#ffffff' : '#8a8371';
            c.textContent = B[byy][bxx];

            c.addEventListener('click', function(){
              B[byy][bxx] = 1 - B[byy][bxx];
              renderB();
              render();
            });
            gB.appendChild(c);
          })(by, bx);
        }
      }
    }

    function render() {
      var g = erode(pixels, B);
      gO.innerHTML = '';
      gN.innerHTML = '';

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          (function(yy, xx){
            var p = pixels[yy][xx] ? 255 : 30;
            var fgO = p > 128 ? '#000000' : '#ffffff';
            var cO = document.createElement('div');
            cO.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;cursor:pointer;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgO + ';box-sizing:border-box;';
            cO.textContent = pixels[yy][xx];

            cO.addEventListener('click', function(){
              pixels[yy][xx] = 1 - pixels[yy][xx];
              render();
            });
            gO.appendChild(cO);
          })(y, x);

          var r = g[y][x] ? 255 : 30;
          var fgN = r > 128 ? '#000000' : '#ffffff';
          var cN = document.createElement('div');
          cN.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + r + ',' + r + ',' + r + ');color:' + fgN + ';box-sizing:border-box;';
          cN.textContent = g[y][x];
          gN.appendChild(cN);
        }
      }
    }

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    btnCross.addEventListener('click', function(){
      B = [[0, 1, 0], [1, 1, 1], [0, 1, 0]];
      renderB();
      render();
    });

    btnBox.addEventListener('click', function(){
      B = [[1, 1, 1], [1, 1, 1], [1, 1, 1]];
      renderB();
      render();
    });

    btnDiag.addEventListener('click', function(){
      B = [[1, 0, 1], [0, 1, 0], [1, 0, 1]];
      renderB();
      render();
    });

    generate();
    renderB();
    render();
  }

  function tryInitSimEP0404(){
    var root = document.getElementById('sim-ep0404-erosao');
    if (root) initSimEP0404(root); else setTimeout(tryInitSimEP0404, 200);
  }
  tryInitSimEP0404();
})();
</script>
</div>
""")

**Figura 4.4:** Simulateur EP04_04: Érosion Binaire Plane (g = f ⊖ B)


<figure id="fig-04-sim-ep0404-erosao">
  <img src="imagens/fig-04-sim-ep0404-erosao.png" alt=" Simulateur EP04_04: Érosion Binaire Plane (g = f ⊖ B) " style="max-width:80%" />
  <figcaption><strong>Figura 4.4:</strong>  Simulateur EP04_04: Érosion Binaire Plane (g = f ⊖ B) </figcaption>
</figure>

In [ ]:
%%writefile EP04_04.py
# Code Python

In [ ]:
TestSuite("EP04_04.py").run()

### EP04_05 🧹 Ouverture Morphologique (Suppression de Bruit)

Les images capturées par des **capteurs à faible coût**, comme ceux des drones agricoles, sont souvent parsemées de petits points de bruit — des pixels isolés qui ne représentent rien de réel. Appliquer une érosion suivie d'une dilatation avec le **même** élément structurant produit l'**ouverture** : elle « nettoie » les points et les fines protubérances, mais rend à l'objet principal pratiquement sa taille d'origine. C'est la combinaison classique utilisée dans le **pré-traitement d'images satellitaires** avant tout comptage de surface cultivée.
Voir dans [Figura 4.5](#fig-04-sim-ep0405-abertura) une simulation de cet EP.

#### 📋 Directives d'Implémentation

1. **Dimensions de l'image :** Lire les entiers $L$ (lignes) et $C$ (colonnes) de $f$.
2. **Dimensions de $B$ :** Lire les entiers $L_B$ (lignes) et $C_B$ (colonnes) de l'élément structurant.
3. **Élément structurant :** Lire la matrice $B$ avec des valeurs $0$ ou $1$, ligne par ligne.
4. **Données :** Lire la matrice binaire $f$ (valeurs $0$ ou $1$), ligne par ligne.
5. **Érosion :** Calculer $e = f \ominus B$, en utilisant exactement l'algorithme de l'EP04_04 (sans réfléchir $B$, sans padding).
6. **Dilatation :** Calculer $g = e \oplus B$, en utilisant exactement l'algorithme de l'EP04_03 (en réfléchissant $B$, sans padding) — mais maintenant appliqué sur $e$, pas sur $f$.
7. **Sortie :** Afficher la matrice résultante $g$ (l'**ouverture** de $f$ par $B$) avec les dimensions $L \times C$.

#### 📌 Contraintes Computationnelles

* **Ordre fixe :** C'est **toujours** l'érosion d'abord, puis la dilatation — l'ordre inverse définit un autre opérateur (la fermeture, du prochain EP).
* **Même $B$ :** L'élément structurant utilisé pour l'érosion et la dilatation doit être identique.
* **Sans padding dans aucune des deux étapes.**

#### 🧠 Fondement Théorique

| Concept | Signification | Impact Visuel |
|----------|-------------|-----------------|
| **Anti-extensivité** | $g \subseteq f$ toujours | L'ouverture ne crée jamais de nouveau pixel, elle ne fait que supprimer |
| **Idempotence** | $\text{ouverture}(\text{ouverture}(f)) = \text{ouverture}(f)$ | Réappliquer ne change plus rien |
| **Points isolés** | Plus petits que $B$ | Ils sont complètement éliminés |
| **Noyau de l'objet** | Plus grand que $B$ | Il est récupéré presque intact par la dilatation finale |

#### 📦 Spécification d'Entrée et de Sortie (VPL)

**Entrée :**

* Ligne 1 : Entier $L$.
* Ligne 2 : Entier $C$.
* Ligne 3 : Entier $L_B$.
* Ligne 4 : Entier $C_B$.
* Les $L_B$ lignes suivantes : éléments entiers ($0$ ou $1$) de la matrice $B$.
* Les $L$ lignes suivantes : éléments entiers ($0$ ou $1$) de la matrice $f$.

**Sortie :**

* Matrice résultante en $L$ lignes et $C$ colonnes, valeurs $0$ ou $1$.

#### 📌 Exemples

| Entrée | Sortie | Observation |
|---------|-------|------------|
| 7<br>7<br>3<br>3<br>1 1 1<br>1 1 1<br>1 1 1<br>0 0 0 0 0 0 0<br>0 1 0 0 0 1 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 1 0<br>0 0 0 0 0 0 0<br>0 1 0 0 0 0 1 | 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 0 0<br>0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 | Les points isolés et la fine protubérance disparaissent ; le carré central survit |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0405-abertura" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧹 Simulateur EP04_05 : Ouverture morphologique</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = (f ⊖ B) ⊕ B</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Cliquez sur les cellules de <b>f original</b> pour allumer ou éteindre des pixels (créez votre propre bruit de fond !) et ajustez la taille de l'élément structurant B.</p>

    <!-- Controle do Tamanho de B -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;text-align:center;">
      <label style="font-size:11px;font-weight:700;color:#8e44ad;">Taille de B (Boîte n×n)</label><br>
      <input type="range" id="sim_ep0405_sl_n" min="3" max="5" step="2" value="3" style="width:60%;cursor:pointer;accent-color:#8e44ad;margin-top:6px;">
      <span id="sim_ep0405_vl_n" style="font-family:monospace;font-size:12px;font-weight:700;color:#8e44ad;margin-left:8px;">3×3</span>
    </div>

    <!-- Pipeline em 3 Colunas: f original vs e (Erosão) vs g (Abertura Final) -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(180px, 1fr));gap:12px;align-items:start;margin-bottom:14px;">
      
      <!-- f Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">f Original (Cliquable)</span>
        <div id="sim_ep0405_grid_f" style="display:grid;grid-template-columns:repeat(7, 28px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- e = f ⊖ B -->
      <div style="background:#fafaf7;border:1px solid #c0392b;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">e = f ⊖ B (Érosion)</span>
        <div id="sim_ep0405_grid_e" style="display:grid;grid-template-columns:repeat(7, 28px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- g = e ⊕ B -->
      <div style="background:#fafaf7;border:2px solid #8e44ad;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#8e44ad;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">g = e ⊕ B (Ouverture)</span>
        <div id="sim_ep0405_grid_g" style="display:grid;grid-template-columns:repeat(7, 28px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Botões de Ação -->
    <div style="text-align:center;display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
      <button id="sim_ep0405_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nouvelle image (avec bruit)</button>
      <button id="sim_ep0405_btnClear" style="padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">🧹 Tout effacer</button>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0405(root){
    if (!root || root.dataset.simEp0405Init) return;
    root.dataset.simEp0405Init = "1";

    var slN     = root.querySelector('#sim_ep0405_sl_n');
    var vlN     = root.querySelector('#sim_ep0405_vl_n');
    var gF      = root.querySelector('#sim_ep0405_grid_f');
    var gE      = root.querySelector('#sim_ep0405_grid_e');
    var gG      = root.querySelector('#sim_ep0405_grid_g');
    var btnNew  = root.querySelector('#sim_ep0405_btnNew');
    var btnClear= root.querySelector('#sim_ep0405_btnClear');

    var L = 7, C = 7, f = [];

    function generate() {
      f = [];
      for (var y = 0; y < L; y++) {
        var row = [];
        for (var x = 0; x < C; x++) row.push(0);
        f.push(row);
      }
      for (var y = 2; y < 5; y++) {
        for (var x = 2; x < 5; x++) f[y][x] = 1;
      }
      for (var k = 0; k < 3; k++) {
        var ry = Math.floor(Math.random() * L), rx = Math.floor(Math.random() * C);
        if (f[ry][rx] === 0 && (ry < 1 || ry > 5 || rx < 1 || rx > 5)) f[ry][rx] = 1;
      }
    }

    function clearAll() {
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) f[y][x] = 0;
      }
    }

    function box(n) {
      var B = [];
      for (var i = 0; i < n; i++) {
        var row = [];
        for (var j = 0; j < n; j++) row.push(1);
        B.push(row);
      }
      return B;
    }

    function reflect(M) {
      var n = M.length, out = [];
      for (var i = n - 1; i >= 0; i--) out.push(M[i].slice().reverse());
      return out;
    }

    function erode(img, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = [];
      for (var y = 0; y < L; y++) g.push(img[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bm[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] < g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function dilate(img, Bm) {
      var Bref = reflect(Bm);
      var HB = Bref.length, WB = Bref[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = [];
      for (var y = 0; y < L; y++) g.push(img[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bref[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] > g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function paintStatic(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          c.style.cssText = 'width:28px;height:28px;border-radius:4px;border:1px solid #e4dcc8;box-sizing:border-box;';
          c.style.background = img[y][x] === 1 ? '#8e44ad' : '#fafaf7';
          grid.appendChild(c);
        }
      }
    }

    function paintEditable(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          (function(yy, xx){
            var c = document.createElement('div');
            c.style.cssText = 'width:28px;height:28px;border-radius:4px;border:1px solid #e4dcc8;cursor:pointer;box-sizing:border-box;transition:all 0.1s ease;';
            c.style.background = img[yy][xx] === 1 ? '#8e44ad' : '#fafaf7';

            c.addEventListener('click', function(){
              f[yy][xx] = 1 - f[yy][xx];
              render();
            });
            grid.appendChild(c);
          })(y, x);
        }
      }
    }

    function render() {
      var n = parseInt(slN.value, 10) || 3;
      vlN.textContent = n + '×' + n;
      var B = box(n);
      var e = erode(f, B);
      var g = dilate(e, B);

      paintEditable(gF, f);
      paintStatic(gE, e);
      paintStatic(gG, g);
    }

    slN.addEventListener('input', render);

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    btnClear.addEventListener('click', function(){
      clearAll();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0405(){
    var root = document.getElementById('sim-ep0405-abertura');
    if (root) initSimEP0405(root); else setTimeout(tryInitSimEP0405, 200);
  }
  tryInitSimEP0405();
})();
</script>
</div>
""")

**Figura 4.5:** Simulateur EP04_05 : Ouverture morphologique (g = (f ⊖ B) ⊕ B)


<figure id="fig-04-sim-ep0405-abertura">
  <img src="imagens/fig-04-sim-ep0405-abertura.png" alt=" Simulateur EP04_05 : Ouverture morphologique (g = (f ⊖ B) ⊕ B) " style="max-width:80%" />
  <figcaption><strong>Figura 4.5:</strong>  Simulateur EP04_05 : Ouverture morphologique (g = (f ⊖ B) ⊕ B) </figcaption>
</figure>

In [ ]:
%%writefile EP04_05.py
# Code Python

In [ ]:
TestSuite("EP04_05.py").run()

### EP04_06 🧩 Fermeture Morphologique (Remplissage des Lacunes)

Dans la **numérisation d'empreintes digitales**, les sillons de la peau sont parfois interrompus par de la saleté ou un dessèchement, créant de petites lacunes dans la courbe continue qui devrait exister. La **fermeture** — dilatation suivie d'une érosion avec le même élément structurant — est l'opérateur dual de l'ouverture : elle **remplit les petits trous et les renfoncements étroits**, sans modifier significativement le contour externe de l'objet. C'est l'étape standard avant d'extraire le squelette d'une empreinte digitale.
Voir dans [Figura 4.6](#fig-04-sim-ep0406-fechamento) une simulation de cet EP.

#### 📋 Directives d'implémentation

1. **Dimensions de l'image :** Lire les entiers $L$ (lignes) et $C$ (colonnes) de $f$.
2. **Dimensions de $B$ :** Lire les entiers $L_B$ (lignes) et $C_B$ (colonnes) de l'élément structurant.
3. **Élément structurant :** Lire la matrice $B$ avec les valeurs $0$ ou $1$, ligne par ligne.
4. **Données :** Lire la matrice binaire $f$ (valeurs $0$ ou $1$), ligne par ligne.
5. **Dilatation :** Calculer $d = f \oplus B$, en utilisant exactement l'algorithme de l'EP04_03 (réflexion de $B$, sans padding).
6. **Érosion :** Calculer $g = d \ominus B$, en utilisant exactement l'algorithme de l'EP04_04 (sans réflexion de $B$, sans padding) — maintenant appliqué sur $d$, et non sur $f$.
7. **Sortie :** Afficher la matrice résultante $g$ (la **fermeture** de $f$ par $B$) avec les dimensions $L \times C$.

#### 📌 Contraintes de calcul

- **Ordre fixe :** C'est **toujours** la dilatation d'abord, puis l'érosion — l'ordre inverse est l'ouverture de l'EP04_05.
- **Même $B$ :** L'élément structurant utilisé dans la dilatation et dans l'érosion doit être identique.
- **Sans padding à aucune des deux étapes.**

#### 🧠 Fondement théorique

| Concept | Signification | Impact visuel |
|---------|---------------|---------------|
| **Extensivité** | $g \supseteq f$ toujours | La fermeture ne supprime jamais de pixel, elle n'ajoute que |
| **Idempotence** | $\text{fermeture}(\text{fermeture}(f)) = \text{fermeture}(f)$ | Réapplique ne change plus rien |
| **Petits trous** | Plus petits que $B$ | Sont complètement remplis |
| **Dualité** | $\text{fermeture}(f) = \overline{\text{ouverture}(\bar f)}$ | C'est l'ouverture appliquée au « négatif » de l'image |

#### 📦 Spécification d'entrée et de sortie (VPL)

**Entrée :**

* Ligne 1 : Entier $L$.
* Ligne 2 : Entier $C$.
* Ligne 3 : Entier $L_B$.
* Ligne 4 : Entier $C_B$.
* Les $L_B$ lignes suivantes : éléments entiers ($0$ ou $1$) de la matrice $B$.
* Les $L$ lignes suivantes : éléments entiers ($0$ ou $1$) de la matrice $f$.

**Sortie :**

* Matrice résultante en $L$ lignes et $C$ colonnes, valeurs $0$ ou $1$.

#### 📌 Exemples

| Entrée | Sortie | Observation |
|--------|--------|-------------|
| 8<br>8<br>3<br>3<br>1 1 1<br>1 1 1<br>1 1 1<br>0 0 0 0 0 0 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 0 1 1 0 0<br>0 0 1 1 0 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 0 0 0 0 0 0 | 0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0 | Les deux trous internes non adjacents sont totalement remplis |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0406-fechamento" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧩 Simulador EP04_06 : Fermeture morphologique</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = (f ⊕ B) ⊖ B</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Cliquez sur les cellules de <b>f original</b> pour allumer ou éteindre les pixels (remplissez les trous internes !) et ajustez la taille de l'élément structurant B.</p>

    <!-- Controle do Tamanho de B -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;text-align:center;">
      <label style="font-size:11px;font-weight:700;color:#2c7a7b;">Taille de B (Boîte n×n)</label><br>
      <input type="range" id="sim_ep0406_sl_n" min="3" max="5" step="2" value="3" style="width:60%;cursor:pointer;accent-color:#2c7a7b;margin-top:6px;">
      <span id="sim_ep0406_vl_n" style="font-family:monospace;font-size:12px;font-weight:700;color:#2c7a7b;margin-left:8px;">3×3</span>
    </div>

    <!-- Pipeline em 3 Colunas: f original vs d (Dilatação) vs g (Fechamento Final) -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(170px, 1fr));gap:12px;align-items:start;margin-bottom:14px;">
      
      <!-- f Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">f Original (Cliquable)</span>
        <div id="sim_ep0406_grid_f" style="display:grid;grid-template-columns:repeat(8, 26px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- d = f ⊕ B -->
      <div style="background:#fafaf7;border:1px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">d = f ⊕ B (Dilatation)</span>
        <div id="sim_ep0406_grid_d" style="display:grid;grid-template-columns:repeat(8, 26px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- g = d ⊖ B -->
      <div style="background:#fafaf7;border:2px solid #2c7a7b;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2c7a7b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">g = d ⊖ B (Fermeture)</span>
        <div id="sim_ep0406_grid_g" style="display:grid;grid-template-columns:repeat(8, 26px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Botões de Ação -->
    <div style="text-align:center;display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
      <button id="sim_ep0406_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Nouvelle Image (Avec Trous)</button>
      <button id="sim_ep0406_btnClear" style="padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">🧹 Tout Effacer</button>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0406(root){
    if (!root || root.dataset.simEp0406Init) return;
    root.dataset.simEp0406Init = "1";

    var slN     = root.querySelector('#sim_ep0406_sl_n');
    var vlN     = root.querySelector('#sim_ep0406_vl_n');
    var gF      = root.querySelector('#sim_ep0406_grid_f');
    var gD      = root.querySelector('#sim_ep0406_grid_d');
    var gG      = root.querySelector('#sim_ep0406_grid_g');
    var btnNew  = root.querySelector('#sim_ep0406_btnNew');
    var btnClear= root.querySelector('#sim_ep0406_btnClear');

    var L = 8, C = 8, f = [];

    function generate() {
      f = [];
      for (var y = 0; y < L; y++) {
        var row = [];
        for (var x = 0; x < C; x++) row.push(0);
        f.push(row);
      }
      for (var y = 1; y < 7; y++) {
        for (var x = 2; x < 6; x++) f[y][x] = 1;
      }
      f[3][3] = 0;
      f[4][4] = 0;
    }

    function clearAll() {
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) f[y][x] = 0;
      }
    }

    function box(n) {
      var B = [];
      for (var i = 0; i < n; i++) {
        var row = [];
        for (var j = 0; j < n; j++) row.push(1);
        B.push(row);
      }
      return B;
    }

    function reflect(M) {
      var n = M.length, out = [];
      for (var i = n - 1; i >= 0; i--) out.push(M[i].slice().reverse());
      return out;
    }

    function dilate(img, Bm) {
      var Bref = reflect(Bm);
      var HB = Bref.length, WB = Bref[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = [];
      for (var y = 0; y < L; y++) g.push(img[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bref[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] > g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function erode(img, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = [];
      for (var y = 0; y < L; y++) g.push(img[y].slice());

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bm[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] < g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function paintStatic(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          c.style.cssText = 'width:26px;height:26px;border-radius:4px;border:1px solid #e4dcc8;box-sizing:border-box;';
          c.style.background = img[y][x] === 1 ? '#2c7a7b' : '#fafaf7';
          grid.appendChild(c);
        }
      }
    }

    function paintEditable(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          (function(yy, xx){
            var c = document.createElement('div');
            c.style.cssText = 'width:26px;height:26px;border-radius:4px;border:1px solid #e4dcc8;cursor:pointer;box-sizing:border-box;transition:all 0.1s ease;';
            c.style.background = img[yy][xx] === 1 ? '#2c7a7b' : '#fafaf7';

            c.addEventListener('click', function(){
              f[yy][xx] = 1 - f[yy][xx];
              render();
            });
            grid.appendChild(c);
          })(y, x);
        }
      }
    }

    function render() {
      var n = parseInt(slN.value, 10) || 3;
      vlN.textContent = n + '×' + n;
      var B = box(n);
      var d = dilate(f, B);
      var g = erode(d, B);

      paintEditable(gF, f);
      paintStatic(gD, d);
      paintStatic(gG, g);
    }

    slN.addEventListener('input', render);

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    btnClear.addEventListener('click', function(){
      clearAll();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0406(){
    var root = document.getElementById('sim-ep0406-fechamento');
    if (root) initSimEP0406(root); else setTimeout(tryInitSimEP0406, 200);
  }
  tryInitSimEP0406();
})();
</script>
</div>
""")

**Figura 4.6:** Simulateur EP04_06: Fermeture morphologique (g = (f ⊕ B) ⊖ B)


<figure id="fig-04-sim-ep0406-fechamento">
  <img src="imagens/fig-04-sim-ep0406-fechamento.png" alt=" Simulateur EP04_06: Fermeture morphologique (g = (f ⊕ B) ⊖ B) " style="max-width:80%" />
  <figcaption><strong>Figura 4.6:</strong>  Simulateur EP04_06: Fermeture morphologique (g = (f ⊕ B) ⊖ B) </figcaption>
</figure>

In [ ]:
%%writefile EP04_06.py
# Code Python

In [ ]:
TestSuite("EP04_06.py").run()

### EP04_07 ⛰️ Dilatation et Érosion Pondérées (mm.dil1 / mm.ero1)

Jusqu'à présent, l'élément structurant disait simplement « ce voisin compte » ou « ne compte pas » — mais dans les **modèles numériques de terrain** (utilisés en SIG et en planification du drainage urbain), chaque voisin devrait avoir un **poids différent** selon la distance ou la direction du relief. Les versions **pondérées** de la dilatation et de l'érosion, implémentées dans `morph.py` comme `mm.dil1(f, b)` et `mm.ero1(f, b)`, additionnent (ou soustraient) le poids de chaque voisin avant de prendre le maximum (ou le minimum) — généralisant ainsi tout ce qui a été fait dans les EP précédents.
Voir dans [Figura 4.7](#fig-04-sim-ep0407-pesos) une simulation de cet EP.

#### 📋 Directives d'Implémentation

1. **Dimensions de l'image :** Lire les entiers $L$ (lignes) et $C$ (colonnes) de $f$.
2. **Dimensions de $b$ :** Lire les entiers $L_B$ (lignes) et $C_B$ (colonnes) de l'élément structurant pondéré.
3. **Poids :** Lire la matrice $b$ des poids **entiers** (ils peuvent être négatifs, nuls ou positifs), ligne par ligne.
4. **Données :** Lire la matrice $f$ (l'image d'origine), ligne par ligne.
5. **Voisinage sans padding :** Pour chaque pixel $(y,x)$, parcourir **toutes** les positions $(by,bx)$ de $b$ (pas seulement celles où la valeur serait $1$ — ici **tout** poids participe), en utilisant le même décalage que dans les EP précédents :
$$
v_y = y + by + o_y,\quad v_x = x + bx + o_x,\quad o_y=-\tfrac{L_B}{2}+0{,}5,\quad o_x=-\tfrac{C_B}{2}+0{,}5
$$
**Écarter** tout $(v_y,v_x)$ hors de $[0,L)\times[0,C)$.
6. **Dilatation pondérée :** Calculer
$$
g_{dil}(y,x) = \max\Big(f(y,x),\ \max_{(v_y,v_x)\ \text{valide}} \big(f(v_y,v_x) + b(by,bx)\big)\Big)
$$
7. **Érosion pondérée :** Calculer, **en utilisant le même $b$ et sans réflexion** :
$$
g_{ero}(y,x) = \min\Big(f(y,x),\ \min_{(v_y,v_x)\ \text{valide}} \big(f(v_y,v_x) - b(by,bx)\big)\Big)
$$
8. **Sortie :** Afficher **d'abord** la matrice complète $g_{dil}$, puis **ensuite** la matrice complète $g_{ero}$.

#### 📌 Contraintes Computationnelles

* **Aucune des deux ne réfléchit $b$** — la version pondérée n'utilise pas la réflexion, même pour la dilatation (contrairement à `mm.dil0`).
* **Tous les poids participent :** Il n'existe pas ici de filtre « $B=1$ » ; même un poids $0$ entre en compte.
* **Sans padding :** les voisins hors de l'image sont ignorés, jamais virtuellement remplis.
* **Type :** La sortie peut contenir des valeurs négatives ou supérieures à $255$ — **pas** de *clipping* dans cet EP.
* **Astuce :** Pour supprimer les messages de dépassement lors du dépassement des limites du type uint8, inclure au début du code :
```python
import warnings
warnings.filterwarnings("ignore")
```

#### 🧠 Fondement Théorique

| Concept | Signification | Impact Visuel |
|----------|---------------|-----------------|
| **Poids positif** | « Tire » la valeur du voisin vers le haut lors de la dilatation | Simule un relief qui monte dans cette direction |
| **Poids négatif** | Réduit la contribution du voisin | Simule la distance ou une atténuation directionnelle |
| **Dualité pondérée** | $\text{ero1}(f,b) = -\text{dil1}(-f,b)$ | La symétrie entre les deux opérations se maintient même avec des poids |

#### 📦 Spécification d'Entrée et de Sortie (VPL)

**Entrée :**

* Ligne 1 : Entier $L$.
* Ligne 2 : Entier $C$.
* Ligne 3 : Entier $L_B$.
* Ligne 4 : Entier $C_B$.
* Les $L_B$ lignes suivantes : éléments entiers (pouvant être négatifs) de la matrice $b$.
* Les $L$ lignes suivantes : éléments entiers de la matrice $f$.

**Sortie :**

* D'abord la matrice $g_{dil}$ en $L$ lignes et $C$ colonnes.
* Ensuite la matrice $g_{ero}$ en $L$ lignes et $C$ colonnes.

#### 📌 Exemples

| Entrée | Sortie | Observation |
|---------|-------|------------|
| 3<br>3<br>3<br>3<br>0 1 0<br>1 2 1<br>0 1 0<br>10 20 30<br>40 50 60<br>70 80 90 | 50 60 61<br>80 90 91<br>81 91 92<br>8 9 19<br>9 10 20<br>39 40 50 | Le poids central $2$ accélère la croissance lors de la dilatation et le rétrécissement lors de l'érosion |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0407-pesos" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">⛰️ Simulateur EP04_07 : Poids dans l'Élément Structurant</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">dil1 / ero1</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Ajustez les poids de l'élément structurant b avec les curseurs et observez l'effet de la dilatation et de l'érosion pondérées sur la matrice f.</p>

    <!-- Painel dos Pesos b -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:14px;margin-bottom:16px;text-align:center;">
      <span style="font-size:10px;font-weight:700;color:#d35400;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Poids b (Ajustez les Curseurs par Cellule)</span>
      <div id="sim_ep0407_grid_b" style="display:grid;grid-template-columns:repeat(3, 70px);gap:8px;justify-content:center;user-select:none;"></div>
    </div>

    <!-- Comparativo em 3 Colunas: f original vs dil1 vs ero1 -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(180px, 1fr));gap:14px;align-items:start;margin-bottom:14px;">
      
      <!-- f Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">f Original</span>
        <div id="sim_ep0407_grid_f" style="display:grid;grid-template-columns:repeat(3, 52px);gap:4px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- dil1(f,b) -->
      <div style="background:#fafaf7;border:1px solid #27ae60;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">dil1(f, b) (Dilatation)</span>
        <div id="sim_ep0407_grid_d" style="display:grid;grid-template-columns:repeat(3, 52px);gap:4px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- ero1(f,b) -->
      <div style="background:#fafaf7;border:1px solid #c0392b;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">ero1(f, b) (Érosion)</span>
        <div id="sim_ep0407_grid_e" style="display:grid;grid-template-columns:repeat(3, 52px);gap:4px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0407(root){
    if (!root || root.dataset.simEp0407Init) return;
    root.dataset.simEp0407Init = "1";

    var gB = root.querySelector('#sim_ep0407_grid_b');
    var gF = root.querySelector('#sim_ep0407_grid_f');
    var gD = root.querySelector('#sim_ep0407_grid_d');
    var gE = root.querySelector('#sim_ep0407_grid_e');

    var b = [[0, 1, 0], [1, 2, 1], [0, 1, 0]];
    var f = [[10, 20, 30], [40, 50, 60], [70, 80, 90]];
    var L = 3, C = 3;

    function compute() {
      var oy = -3 / 2 + 0.5, ox = -3 / 2 + 0.5;
      var dil = f.map(function(r){ return r.slice(); });
      var ero = f.map(function(r){ return r.slice(); });

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < 3; by++) {
            for (var bx = 0; bx < 3; bx++) {
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C) {
                var cd = f[vy][vx] + b[by][bx];
                if (cd > dil[y][x]) dil[y][x] = cd;
                var ce = f[vy][vx] - b[by][bx];
                if (ce < ero[y][x]) ero[y][x] = ce;
              }
            }
          }
        }
      }
      return { dil: dil, ero: ero };
    }

    function renderB() {
      gB.innerHTML = '';
      for (var by = 0; by < 3; by++) {
        for (var bx = 0; bx < 3; bx++) {
          (function(row, col){
            var wrap = document.createElement('div');
            wrap.style.cssText = 'display:flex;flex-direction:column;align-items:center;background:#fafaf7;border:1px solid #e4dcc8;border-radius:6px;padding:4px;box-sizing:border-box;';

            var val = document.createElement('div');
            val.style.cssText = 'font-family:monospace;font-weight:700;font-size:11px;color:#d35400;margin-bottom:2px;';
            val.textContent = b[row][col];

            var sl = document.createElement('input');
            sl.type = 'range';
            sl.min = '-5';
            sl.max = '5';
            sl.step = '1';
            sl.value = b[row][col];
            sl.style.cssText = 'width:56px;cursor:pointer;accent-color:#d35400;';

            sl.addEventListener('input', function(){
              b[row][col] = parseInt(sl.value, 10);
              val.textContent = b[row][col];
              renderAll();
            });

            wrap.appendChild(val);
            wrap.appendChild(sl);
            gB.appendChild(wrap);
          })(by, bx);
        }
      }
    }

    function paint(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          var v = img[y][x];
          var inten = Math.min(255, Math.max(0, v));
          var fg = inten > 128 ? '#000000' : '#ffffff';
          c.style.cssText = 'width:52px;height:42px;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + inten + ',' + inten + ',' + inten + ');color:' + fg + ';box-sizing:border-box;';
          c.textContent = v;
          grid.appendChild(c);
        }
      }
    }

    function renderAll() {
      var res = compute();
      paint(gF, f);
      paint(gD, res.dil);
      paint(gE, res.ero);
    }

    renderB();
    renderAll();
  }

  function tryInitSimEP0407(){
    var root = document.getElementById('sim-ep0407-pesos');
    if (root) initSimEP0407(root); else setTimeout(tryInitSimEP0407, 200);
  }
  tryInitSimEP0407();
})();
</script>
</div>
""")

**Figura 4.7:** Simulateur EP04_07 : Dilatation et Érosion avec Poids (mm.dil1 / mm.ero1)


<figure id="fig-04-sim-ep0407-pesos">
  <img src="imagens/fig-04-sim-ep0407-pesos.png" alt=" Simulateur EP04_07 : Dilatation et Érosion avec Poids (mm.dil1 / mm.ero1) " style="max-width:80%" />
  <figcaption><strong>Figura 4.7:</strong>  Simulateur EP04_07 : Dilatation et Érosion avec Poids (mm.dil1 / mm.ero1) </figcaption>
</figure>

In [ ]:
%%writefile EP04_07.py
# Code Python

In [ ]:
TestSuite("EP04_07.py").run()

### EP04_08 🌋 Gradient morphologique, Top-hat et Black-hat

En **inspection automatique de plaques de circuits**, trois questions reviennent constamment : où se trouvent les **bords** des composants ? Quels **détails clairs et petits** (comme les points de soudure) se détachent du fond ? Quelles **cavités sombres** (comme les fissures) le fond dissimule-t-il ? Une seule paire érosion/dilatation répond aux trois : le **gradient morphologique** met en évidence les contours, le **top-hat** révèle les pics étroits, et le **black-hat** révèle les vallées étroites — trois outils, un seul voisinage.
Voir dans [Figura 4.8](#fig-04-sim-ep0408-gradiente) une simulation de cet EP.

#### 📋 Directives d'implémentation

1. **Dimensions de l'image :** Lire les entiers $L$ (lignes) et $C$ (colonnes) de $f$.
2. **Dimensions de $B$ :** Lire les entiers $L_B$ (lignes) et $C_B$ (colonnes) de l'élément structurant.
3. **Élément structurant :** Lire la matrice $B$ avec des valeurs $0$ ou $1$, ligne par ligne.
4. **Données :** Lire la matrice $f$ (l'image originale, en niveaux de gris), ligne par ligne.
5. **Opérateurs de base :** Calculer, exactement comme dans les EP 04_03 à 04_06 :
   * $d = f \oplus B$ (dilatation),
   * $e = f \ominus B$ (érosion),
   * $\text{ouverture} = e \oplus B$,
   * $\text{fermeture} = d \ominus B$.
6. **Gradient morphologique :** $\text{grad}(y,x) = d(y,x) - e(y,x)$.
7. **Top-hat :** $\text{tophat}(y,x) = f(y,x) - \text{ouverture}(y,x)$.
8. **Black-hat :** $\text{blackhat}(y,x) = \text{fermeture}(y,x) - f(y,x)$.
9. **Sortie :** Afficher, **dans cet ordre**, les trois matrices complètes : gradient, top-hat, black-hat.

#### 📌 Contraintes computationnelles

* **Sans padding à aucune étape intermédiaire** — dilatation, érosion, ouverture et fermeture suivent les mêmes règles de voisinage que les EP précédents.
* **Pas de *clipping* :** les trois sorties peuvent contenir n'importe quelle valeur entière (le gradient est toujours $\geq 0$, mais top-hat et black-hat le sont aussi).
* **Réutilisation :** $d$ et $e$ doivent être calculés **une seule fois** et réutilisés pour construire ouverture, fermeture et gradient.

#### 🧠 Fondements théoriques

| Opérateur | Formule | Ce qu'il révèle |
|----------|---------|----------------|
| **Gradient** | $d - e$ | Bords : zéro dans les régions planes, élevé aux transitions |
| **Top-hat** | $f - \text{ouverture}(f)$ | Éléments **clairs et fins**, plus petits que $B$ |
| **Black-hat** | $\text{fermeture}(f) - f$ | Éléments **sombres et fins**, plus petits que $B$ |

#### 📦 Spécification d'entrée et de sortie (VPL)

**Entrée :**

* Ligne 1 : Entier $L$.
* Ligne 2 : Entier $C$.
* Ligne 3 : Entier $L_B$.
* Ligne 4 : Entier $C_B$.
* Lignes suivantes $L_B$ : éléments entiers ($0$ ou $1$) de la matrice $B$.
* Lignes suivantes $L$ : éléments entiers de la matrice $f$.

**Sortie :**

* Matrice gradient en $L$ lignes et $C$ colonnes.
* Matrice top-hat en $L$ lignes et $C$ colonnes.
* Matrice black-hat en $L$ lignes et $C$ colonnes.

#### 📌 Exemples

| Entrée | Sortie | Observation |
|---------|-------|------------|
| 9<br>9<br>3<br>3<br>1 1 1<br>1 1 1<br>1 1 1<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 80 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 2 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10 | (gradient : halo $3\times3=70$ autour de $(2,2)$ et halo $3\times3=8$ autour de $(6,6)$, reste $0$)<br>(top-hat : unique $70$ en $(2,2)$, reste $0$)<br>(black-hat : unique $8$ en $(6,6)$, reste $0$) | Pic isolé devient top-hat ; vallée isolée devient black-hat ; les deux apparaissent dans le gradient |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0408-gradiente" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🌋 Simulateur EP04_08 : Gradient / Top-hat / Black-hat</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">3 opérateurs, 1 voisinage</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Ajoutez des pics ou des creux dans la matrice f et observez le comportement simultané des opérateurs de gradient, top-hat et black-hat.</p>

    <!-- Botões de Ação -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;text-align:center;display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
      <button id="sim_ep0408_btn_pico" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #b9770e;background:#fef5e7;color:#b9770e;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">☀️ Ajouter un Pic</button>
      <button id="sim_ep0408_btn_vale" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #2980b9;background:#ebf4fd;color:#2980b9;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">🕳️ Ajouter un Creux</button>
      <button id="sim_ep0408_btn_reset" style="padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">↩ Tout Effacer</button>
    </div>

    <!-- Comparativo em 4 Colunas: f, Gradiente, Top-hat, Black-hat -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(140px, 1fr));gap:12px;align-items:start;margin-bottom:14px;">
      
      <!-- Matriz f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:10px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#7f8c8d;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">f (Entrée)</span>
        <div id="sim_ep0408_grid_f" style="display:grid;grid-template-columns:repeat(9, 20px);gap:1px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Gradiente -->
      <div style="background:#fafaf7;border:1px solid #8e44ad;border-radius:12px;padding:10px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#8e44ad;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Gradient</span>
        <div id="sim_ep0408_grid_grad" style="display:grid;grid-template-columns:repeat(9, 20px);gap:1px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Top-hat -->
      <div style="background:#fafaf7;border:1px solid #d35400;border-radius:12px;padding:10px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#d35400;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Top-hat</span>
        <div id="sim_ep0408_grid_th" style="display:grid;grid-template-columns:repeat(9, 20px);gap:1px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Black-hat -->
      <div style="background:#fafaf7;border:1px solid #2980b9;border-radius:12px;padding:10px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Black-hat</span>
        <div id="sim_ep0408_grid_bh" style="display:grid;grid-template-columns:repeat(9, 20px);gap:1px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0408(root){
    if (!root || root.dataset.simEp0408Init) return;
    root.dataset.simEp0408Init = "1";

    var gF   = root.querySelector('#sim_ep0408_grid_f');
    var gGrad= root.querySelector('#sim_ep0408_grid_grad');
    var gTh  = root.querySelector('#sim_ep0408_grid_th');
    var gBh  = root.querySelector('#sim_ep0408_grid_bh');

    var btnPico  = root.querySelector('#sim_ep0408_btn_pico');
    var btnVale  = root.querySelector('#sim_ep0408_btn_vale');
    var btnReset = root.querySelector('#sim_ep0408_btn_reset');

    var L = 9, C = 9, f = [], B = [[1, 1, 1], [1, 1, 1], [1, 1, 1]];

    function resetMatrix() {
      f = Array.from({ length: L }, function(){ return new Array(C).fill(10); });
    }

    function morph(img, Bm, mode) {
      var Bref = mode === 'dil' ? Bm.slice().reverse().map(function(r){ return r.slice().reverse(); }) : Bm;
      var HB = Bref.length, WB = Bref[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = img.map(function(r){ return r.slice(); });

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bref[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C) {
                if (mode === 'dil' && img[vy][vx] > g[y][x]) g[y][x] = img[vy][vx];
                if (mode === 'ero' && img[vy][vx] < g[y][x]) g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function paint(grid, img, cmin, cmax, hue) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var v = img[y][x];
          var t = cmax > cmin ? (v - cmin) / (cmax - cmin) : 0;
          var c = document.createElement('div');
          c.style.cssText = 'width:20px;height:20px;border-radius:3px;box-sizing:border-box;';
          c.style.background = v === 0 ? '#fafaf7' : hue;
          c.style.opacity = v === 0 ? '1' : (0.35 + 0.65 * Math.min(1, t));
          grid.appendChild(c);
        }
      }
    }

    function render() {
      var d = morph(f, B, 'dil');
      var e = morph(f, B, 'ero');
      var ab = morph(e, B, 'dil');
      var fc = morph(d, B, 'ero');

      var grad = f.map(function(r, y){ return r.map(function(_, x){ return d[y][x] - e[y][x]; }); });
      var th   = f.map(function(r, y){ return r.map(function(v, x){ return v - ab[y][x]; }); });
      var bh   = f.map(function(r, y){ return r.map(function(v, x){ return fc[y][x] - v; }); });

      var maxF = Math.max.apply(null, f.map(function(r){ return Math.max.apply(null, r); }));
      var maxG = Math.max.apply(null, grad.map(function(r){ return Math.max.apply(null, r); }));
      var maxTh = Math.max.apply(null, th.map(function(r){ return Math.max.apply(null, r); }));
      var maxBh = Math.max.apply(null, bh.map(function(r){ return Math.max.apply(null, r); }));

      paint(gF, f, 10, maxF || 1, '#7f8c8d');
      paint(gGrad, grad, 0, Math.max(1, maxG), '#8e44ad');
      paint(gTh, th, 0, Math.max(1, maxTh), '#d35400');
      paint(gBh, bh, 0, Math.max(1, maxBh), '#2980b9');
    }

    btnPico.addEventListener('click', function(){
      var y = 2 + Math.floor(Math.random() * 5), x = 2 + Math.floor(Math.random() * 5);
      f[y][x] = Math.min(255, f[y][x] + 60 + Math.floor(Math.random() * 30));
      render();
    });

    btnVale.addEventListener('click', function(){
      var y = 2 + Math.floor(Math.random() * 5), x = 2 + Math.floor(Math.random() * 5);
      f[y][x] = Math.max(0, f[y][x] - 8 - Math.floor(Math.random() * 4));
      render();
    });

    btnReset.addEventListener('click', function(){
      resetMatrix();
      f[2][2] = 80;
      f[6][6] = 2;
      render();
    });

    resetMatrix();
    f[2][2] = 80;
    f[6][6] = 2;
    render();
  }

  function tryInitSimEP0408(){
    var root = document.getElementById('sim-ep0408-gradiente');
    if (root) initSimEP0408(root); else setTimeout(tryInitSimEP0408, 200);
  }
  tryInitSimEP0408();
})();
</script>
</div>
""")

**Figura 4.8:** Simulateur EP04_08: Gradient morphologique, Top-hat et Black-hat


<figure id="fig-04-sim-ep0408-gradiente">
  <img src="imagens/fig-04-sim-ep0408-gradiente.png" alt=" Simulateur EP04_08: Gradient morphologique, Top-hat et Black-hat " style="max-width:80%" />
  <figcaption><strong>Figura 4.8:</strong>  Simulateur EP04_08: Gradient morphologique, Top-hat et Black-hat </figcaption>
</figure>

In [ ]:
%%writefile EP04_08.py
# Code Python

In [ ]:
TestSuite("EP04_08.py").run()

### EP04_09 🗺️ Transformée de distance et le « cœur » de l’objet

En **robotique mobile**, lors de la planification d’un itinéraire dans un couloir, le robot souhaite savoir non seulement *où* se trouve l’espace libre, mais aussi **à quelle distance** chaque point libre se trouve du mur le plus proche. Les chemins les plus sûrs tendent à passer par le « cœur » du couloir, loin des obstacles.

La **transformée de distance morphologique** attribue à chaque pixel une valeur représentant sa distance jusqu’au bord le plus proche, selon la métrique définie par l’élément structurant. Les pixels proches du bord reçoivent des valeurs faibles, tandis que les pixels plus internes reçoivent des valeurs plus élevées. Le pixel de valeur maximale correspond à la région la plus protégée de l’objet, souvent associée à son centre morphologique.

Voir dans [Figura 4.9](#fig-04-sim-ep0409-distancia) une simulation de cet EP.

#### 📋 Directives d’implémentation

1. **Dimensions de l’image :** lire les entiers $L$ (lignes) et $C$ (colonnes) de l’image $f$.
2. **Dimensions de $B$ :** lire les entiers $L_B$ (lignes) et $C_B$ (colonnes) de l’élément structurant.
3. **Élément structurant :** lire la matrice $b$, contenant la valeur $0$ au centre et des valeurs négatives aux autres positions.
4. **Image :** lire la matrice binaire $f$ (valeurs $0$ ou $1$), ligne par ligne.
5. **Préparation :** multiplier l’image par $L\times C$, en garantissant que les pixels internes ont une valeur initiale suffisamment élevée pour la propagation des distances.
6. **Transformée de distance :** calculer la matrice des distances en utilisant la méthode `mm.dist1(f,b)`.
7. **Sortie :** afficher la matrice résultante de la transformée de distance.

#### 📌 Contraintes de calcul

* Utiliser l’implémentation de l’érosion pondérée fournie par la bibliothèque.
* L’élément structurant peut contenir des valeurs négatives arbitraires.
* La transformée doit être obtenue par l’application itérative d’érosions pondérées jusqu’à atteindre un point fixe.

**⚠️ Note cruciale sur la lecture des matrices :** Comme l’élément structurant peut contenir des entiers négatifs (par exemple, `-1` et `-99`), **ne pas utiliser la fonction `mm.readImg` pour lire la matrice $b$**. Cette fonction convertit les données en type `uint8`, provoquant un *underflow* et corrompant les valeurs négatives. Lire les $L_B$ lignes de $b$ manuellement en utilisant le type standard `int`. L’image $f$ peut continuer à être lue normalement par `mm.readImg`.

#### 🧠 Fondement théorique

| Concept                            | Signification                                                                       | Impact visuel                               |
| ----------------------------------- | ----------------------------------------------------------------------------------- | ------------------------------------------- |
| **$\text{dist}(y,x)$**              | Distance morphologique jusqu’au bord le plus proche selon la métrique définie par $b$ | Les pixels plus internes reçoivent des valeurs plus élevées |
| **Valeur maximale**                 | Pixel le plus éloigné du bord                                                       | Se rapproche du centre morphologique de l’objet |
| **Élément structurant pondéré**     | Définit les coûts de déplacement entre pixels voisins                               | Détermine la métrique de distance utilisée  |
| **Objets fins**                     | Régions étroites de l’objet                                                         | Produisent des valeurs de distance faibles  |

#### 📦 Spécification d’entrée et de sortie (VPL)

**Entrée :**

* Ligne 1 : entier $L$.
* Ligne 2 : entier $C$.
* Ligne 3 : entier $L_B$.
* Ligne 4 : entier $C_B$.
* Les $L_B$ lignes suivantes : éléments entiers de la matrice $b$.
* Les $L$ lignes suivantes : éléments binaires ($0$ ou $1$) de la matrice $f$.

⚠️ **Note d’implémentation :** Les éléments de la matrice $f$ (0 ou 1) doivent être multipliés par **255** pour générer une image binaire appropriée ($0$ et $255$) avant d’appliquer la Transformée de Distance (TD).

**Sortie :**

* Matrice de la transformée de distance en $L$ lignes et $C$ colonnes.

#### 📌 Exemple

| Entrée                                                                                                                                                          | Sortie                                                                                                | Observation                              |
| ---------------------------------------------------------------------------------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------- | ---------------------------------------- |
| 5<br>9<br>3<br>3<br>-99 -1 -99<br>-1 0 -1<br>-99 -1 -99<br>0 0 0 0 0 0 0 0 0<br>0 1 1 1 1 1 1 1 0<br>0 1 1 1 1 1 1 1 0<br>0 1 1 1 1 1 1 1 0<br>0 0 0 0 0 0 0 0 0 | 0 0 0 0 0 0 0 0 0<br>0 1 1 1 1 1 1 1 0<br>0 1 2 2 2 2 2 1 0<br>0 1 1 1 1 1 1 1 0<br>0 0 0 0 0 0 0 0 0 | Résultat de la transformée de distance. |

**Note :** la valeur `-99` agit comme une approximation pratique de $-\infty$, empêchant la propagation par les diagonales. Ainsi, seuls les voisins horizontaux et verticaux contribuent à la distance, produisant la distance de Manhattan.

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0409-distancia" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🗺️ Simulateur EP04_09 : Transformée de distance</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">Couches d'érosion</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Cliquez sur les cellules pour dessiner votre propre objet ou sélectionnez une forme prédéfinie pour calculer la carte des distances en cascade.</p>

    <!-- Grade f Original Clicável -->
    <div style="display:flex;justify-content:center;margin-bottom:14px;">
      <div id="sim_ep0409_grid_f" style="display:grid;grid-template-columns:repeat(9, 32px);gap:2px;user-select:none;"></div>
    </div>

    <!-- Botões de Formas Predefinidas -->
    <div style="text-align:center;margin-bottom:14px;display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
      <button id="sim_ep0409_btn_corredor" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #16a085;background:#eafaf1;color:#16a085;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">📐 Couloir</button>
      <button id="sim_ep0409_btn_disco" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #16a085;background:#eafaf1;color:#16a085;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">⬤ Disque</button>
      <button id="sim_ep0409_btn_l" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #16a085;background:#eafaf1;color:#16a085;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">📏 Forme en L</button>
    </div>

    <!-- Título do Mapa de Distâncias -->
    <span style="font-size:10px;font-weight:700;color:#5e5a4a;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;text-align:center;">Carte des distances calculée</span>

    <!-- Grade de Distâncias -->
    <div style="display:flex;justify-content:center;">
      <div id="sim_ep0409_grid_dist" style="display:grid;grid-template-columns:repeat(9, 32px);gap:2px;user-select:none;"></div>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0409(root){
    if (!root || root.dataset.simEp0409Init) return;
    root.dataset.simEp0409Init = "1";

    var gF = root.querySelector('#sim_ep0409_grid_f');
    var gD = root.querySelector('#sim_ep0409_grid_dist');

    var L = 5, C = 9, f = [];
    var B = [[0, 1, 0], [1, 1, 1], [0, 1, 0]];

    function setCorredor() {
      f = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      for (var y = 1; y < 4; y++) {
        for (var x = 1; x < 8; x++) f[y][x] = 1;
      }
    }

    function setDisco() {
      f = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      var cy = 2, cx = 4;
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          if (Math.pow(y - cy, 2) + Math.pow((x - cx) * 0.6, 2) <= 4) f[y][x] = 1;
        }
      }
    }

    function setL() {
      f = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      for (var y = 1; y < 4; y++) {
        for (var x = 1; x < 3; x++) f[y][x] = 1;
      }
      for (var y = 2; y < 4; y++) {
        for (var x = 1; x < 8; x++) f[y][x] = 1;
      }
    }

    function erode(img, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = img.map(function(r){ return r.slice(); });

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bm[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] < g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function sameMatrix(a, b) {
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          if (a[y][x] !== b[y][x]) return false;
        }
      }
      return true;
    }

    function render() {
      gF.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          (function(yy, xx){
            var c = document.createElement('div');
            c.style.cssText = 'width:32px;height:32px;border-radius:6px;border:1px solid #e4dcc8;cursor:pointer;box-sizing:border-box;transition:all 0.1s ease;';
            c.style.background = f[yy][xx] ? '#16a085' : '#fafaf7';

            c.addEventListener('click', function(){
              f[yy][xx] = 1 - f[yy][xx];
              render();
            });
            gF.appendChild(c);
          })(y, x);
        }
      }

      var dist = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      var atual = f.map(function(r){ return r.slice(); });
      var nivel = 0;

      while (atual.some(function(r){ return r.some(function(v){ return v === 1; }); })) {
        nivel++;
        for (var y = 0; y < L; y++) {
          for (var x = 0; x < C; x++) {
            if (atual[y][x] === 1) dist[y][x] = nivel;
          }
        }
        var prox = erode(atual, B);
        if (sameMatrix(prox, atual)) break;
        atual = prox;
      }

      gD.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          var v = dist[y][x];
          var t = v / (nivel || 1);
          var inten = Math.round(220 - t * 170);

          c.style.cssText = 'width:32px;height:32px;border-radius:6px;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:700;font-family:monospace;border:1px solid #e4dcc8;box-sizing:border-box;';
          c.style.background = v === 0 ? '#fafaf7' : 'rgb(' + (inten - 60) + ',' + inten + ',' + (inten - 30) + ')';
          c.style.color = v > 0 ? '#ffffff' : '#8a8371';
          c.textContent = v || '';
          gD.appendChild(c);
        }
      }
    }

    root.querySelector('#sim_ep0409_btn_corredor').addEventListener('click', function(){ setCorredor(); render(); });
    root.querySelector('#sim_ep0409_btn_disco').addEventListener('click', function(){ setDisco(); render(); });
    root.querySelector('#sim_ep0409_btn_l').addEventListener('click', function(){ setL(); render(); });

    setCorredor();
    render();
  }

  function tryInitSimEP0409(){
    var root = document.getElementById('sim-ep0409-distancia');
    if (root) initSimEP0409(root); else setTimeout(tryInitSimEP0409, 200);
  }
  tryInitSimEP0409();
})();
</script>
</div>
""")

**Figura 4.9:** Simulateur EP04_09: Transformée de Distance (Couches d


<figure id="fig-04-sim-ep0409-distancia">
  <img src="imagens/fig-04-sim-ep0409-distancia.png" alt=" Simulateur EP04_09: Transformée de Distance (Couches d'Érosion) " style="max-width:80%" />
  <figcaption><strong>Figura 4.9:</strong>  Simulateur EP04_09: Transformée de Distance (Couches d'Érosion) </figcaption>
</figure>

In [ ]:
%%writefile EP04_09.py
# Code Python

In [ ]:
TestSuite("EP04_09.py").run()

### EP04_10 🪙 Séparation des *blobs*, étiquetage et descripteurs

Dans une **ligne de production de pièces de monnaie**, il est courant que les pièces se touchent sur le tapis roulant, formant une seule tache connectée dans l'image — un comptage naïf donnerait un nombre erroné. La solution classique combine des opérations morphologiques et une analyse de connectivité : d'abord, une **érosion** réduit ou rompt les connexions fragiles entre les objets, puis l'**étiquetage des composantes connexes** sépare chaque objet en une région distincte. Enfin, des **descripteurs géométriques** (aire et boîte englobante) résument chaque composante trouvée.

Voir [Figura 4.10](#fig-04-sim-ep0410-rotulacao) pour une simulation de cet EP.

#### 📋 Directives d'implémentation

1. **Dimensions de l'image :** lire les entiers $L$ (lignes) et $C$ (colonnes) de $f$.

2. **Dimensions de $B$ :** lire les entiers $L_B$ (lignes) et $C_B$ (colonnes) de l'élément structurant.

3. **Élément structurant :** lire la matrice $B$, contenant des valeurs $0$ ou $1$, ligne par ligne.

4. **Données :** lire la matrice binaire $f$ (valeurs $0$ ou $1$), ligne par ligne.

5. **Séparation :** calculer
   $$
   f_{ero} = f \ominus B
   $$
   en utilisant une érosion binaire plane (comme dans l'EP04_04), en éliminant les connexions fragiles entre les objets.

6. **Étiquetage :** sur $f_{ero}$, identifier les composantes connexes en utilisant la connectivité définie par le voisinage $B$. L'étiquetage doit suivre un balayage *raster* : lorsqu'un pixel $1$ non encore étiqueté est trouvé, attribuer un nouveau label entier croissant à partir de 1 et propager ce label à toute la région connexe.

7. **Descripteurs :** pour chaque label $k$, calculer :

   * **Aire :** nombre de pixels appartenant au label ;
   * **Boîte englobante :** $$(y_{min}, x_{min}, y_{max}, x_{max})$$

8. **Sortie :** afficher le nombre total de labels, puis une ligne par label au format :
   $$
   k,\ \text{aire},\ y_{min},\ x_{min},\ y_{max},\ x_{max}
   $$

#### 📌 Contraintes computationnelles

* L'érosion doit être appliquée avant l'étiquetage.
* La connectivité est fixe et définie par le voisinage ci-dessus.
* L'élément structurant $B$ n'interfère pas avec la connectivité de l'étiquetage.
* Aucun padding à aucune étape.
* L'ordre des labels suit la première découverte en balayage *raster*.

#### 🧠 Fondement théorique

| Concept           | Signification                                 | Impact                                              |
| ----------------- | --------------------------------------------- | --------------------------------------------------- |
| Pont fin          | Connexion étroite entre objets                | Peut être supprimé par l'érosion morphologique       |
| Connectivité      | Définie par l'ensemble $$\mathcal{N}(y,x)$$   | Détermine quels pixels appartiennent à la même composante |
| Aire               | Nombre de pixels par composante               | Estimation directe de la taille de l'objet           |
| Boîte englobante  | Extension spatiale du label                   | Résumé géométrique de la composante                 |

#### 📦 Spécification d'entrée et de sortie (VPL)

**Entrée :**

* Ligne 1 : entier $L$
* Ligne 2 : entier $C$
* Ligne 3 : entier $L_B$
* Ligne 4 : entier $C_B$
* Les $L_B$ lignes suivantes : matrice $B$
* Les $L$ lignes suivantes : matrice $f$

**Sortie :**

* Ligne 1 : nombre total de labels trouvés
* Lignes suivantes :
  $$
  k,\ \text{aire},\ y_{min},\ x_{min},\ y_{max},\ x_{max}
  $$

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0410-rotulacao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🪙 Simulateur EP04_10 : Pièces Collées → Séparées → Comptées</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">érosion + étiquette + descripteurs</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Ajustez l'épaisseur du pont entre les pièces et observez comment l'érosion morphologique sépare les objets pour le comptage et l'extraction de descripteurs (aire et boîte englobante).</p>

    <!-- Controle de Espessura da Ponte -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;text-align:center;">
      <label style="font-size:11px;font-weight:700;color:#b9770e;">Épaisseur du Pont entre les Pièces</label><br>
      <input type="range" id="sim_ep0410_sl_p" min="1" max="3" step="1" value="1" style="width:60%;cursor:pointer;accent-color:#b9770e;margin-top:6px;">
      <span id="sim_ep0410_vl_p" style="font-family:monospace;font-size:12px;font-weight:700;color:#b9770e;margin-left:8px;">1 px</span>
    </div>

    <!-- Comparativo Lado a Lado: f original vs Rótulos Pós-Erosão -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- f Original (ligadas) -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Original (Collées)</span>
        <div id="sim_ep0410_grid_f" style="display:grid;grid-template-columns:repeat(10, 26px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Após Erosão + Rótulos -->
      <div style="background:#fafaf7;border:1px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Après Érosion + Étiquettes</span>
        <div id="sim_ep0410_grid_lab" style="display:grid;grid-template-columns:repeat(10, 26px);gap:2px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Painel Informativo / Descritores -->
    <div id="sim_ep0410_info" style="background:#fef5e7;border:1px solid #f8c471;border-radius:8px;padding:10px 14px;font-size:11px;color:#7d5a00;text-align:center;line-height:1.5;"></div>

  </div>
</div>

<script>
(function(){
  function initSimEP0410(root){
    if (!root || root.dataset.simEp0410Init) return;
    root.dataset.simEp0410Init = "1";

    var slP  = root.querySelector('#sim_ep0410_sl_p');
    var vlP  = root.querySelector('#sim_ep0410_vl_p');
    var gF   = root.querySelector('#sim_ep0410_grid_f');
    var gL   = root.querySelector('#sim_ep0410_grid_lab');
    var info = root.querySelector('#sim_ep0410_info');

    var L = 7, C = 10;
    var B = [[1, 1, 1], [1, 1, 1], [1, 1, 1]];
    var palette = ['#e74c3c', '#27ae60', '#2980b9', '#8e44ad', '#d35400'];

    function buildF(p) {
      var f = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      for (var y = 1; y < 6; y++) {
        for (var x = 1; x < 4; x++) f[y][x] = 1;
      }
      for (var y = 1; y < 6; y++) {
        for (var x = 6; x < 9; x++) f[y][x] = 1;
      }
      var midRow = 3;
      for (var dy = 0; dy < p; dy++) {
        var ry = midRow - Math.floor(p / 2) + dy;
        for (var x = 4; x < 6; x++) f[ry][x] = 1;
      }
      return f;
    }

    function erode(img, Bm) {
      var HB = Bm.length, WB = Bm[0].length, oy = -HB / 2 + 0.5, ox = -WB / 2 + 0.5;
      var g = img.map(function(r){ return r.slice(); });

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          for (var by = 0; by < HB; by++) {
            for (var bx = 0; bx < WB; bx++) {
              if (Bm[by][bx] !== 1) continue;
              var vy = Math.trunc(y + by + oy), vx = Math.trunc(x + bx + ox);
              if (vy >= 0 && vy < L && vx >= 0 && vx < C && img[vy][vx] < g[y][x]) {
                g[y][x] = img[vy][vx];
              }
            }
          }
        }
      }
      return g;
    }

    function labelK8(img) {
      var labels = Array.from({length: L}, function(){ return new Array(C).fill(0); });
      var dirs = [[-1, -1], [-1, 0], [-1, 1], [0, -1], [0, 1], [1, -1], [1, 0], [1, 1]];
      var cur = 0, desc = [];

      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          if (img[y][x] === 1 && labels[y][x] === 0) {
            cur++;
            var stack = [[y, x]];
            labels[y][x] = cur;
            var area = 0, miny = y, maxy = y, minx = x, maxx = x;

            while (stack.length) {
              var cell = stack.pop();
              var cy = cell[0], cx = cell[1];
              area++;
              if (cy < miny) miny = cy;
              if (cy > maxy) maxy = cy;
              if (cx < minx) minx = cx;
              if (cx > maxx) maxx = cx;

              dirs.forEach(function(d){
                var ny = cy + d[0], nx = cx + d[1];
                if (ny >= 0 && ny < L && nx >= 0 && nx < C && img[ny][nx] === 1 && labels[ny][nx] === 0) {
                  labels[ny][nx] = cur;
                  stack.push([ny, nx]);
                }
              });
            }
            desc.push({k: cur, area: area, miny: miny, minx: minx, maxy: maxy, maxx: maxx});
          }
        }
      }
      return {labels: labels, desc: desc};
    }

    function paintBin(grid, img) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          c.style.cssText = 'width:26px;height:26px;border-radius:4px;border:1px solid #e4dcc8;box-sizing:border-box;';
          c.style.background = img[y][x] ? '#b9770e' : '#fafaf7';
          grid.appendChild(c);
        }
      }
    }

    function paintLabels(grid, labels) {
      grid.innerHTML = '';
      for (var y = 0; y < L; y++) {
        for (var x = 0; x < C; x++) {
          var c = document.createElement('div');
          var k = labels[y][x];
          c.style.cssText = 'width:26px;height:26px;border-radius:4px;border:1px solid #e4dcc8;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;color:#ffffff;box-sizing:border-box;';
          c.style.background = k > 0 ? palette[(k - 1) % palette.length] : '#fafaf7';
          c.textContent = k > 0 ? k : '';
          grid.appendChild(c);
        }
      }
    }

    function render() {
      var p = parseInt(slP.value, 10) || 1;
      vlP.textContent = p + ' px';
      var f = buildF(p);
      var fe = erode(f, B);
      var res = labelK8(fe);

      paintBin(gF, f);
      paintLabels(gL, res.labels);

      var txt = '<b>' + res.desc.length + ' objeto(s) detectado(s) após a erosão.</b><br>';
      res.desc.forEach(function(d){
        txt += 'Rótulo ' + d.k + ': área = ' + d.area + ', bbox = (' + d.miny + ',' + d.minx + ') → (' + d.maxy + ',' + d.maxx + ')<br>';
      });
      if (res.desc.length < 2) {
        txt += '<i>A ponte ainda é espessa demais para a erosão 3×3 — as moedas continuam fundidas em 1 só objeto.</i>';
      }
      info.innerHTML = txt;
    }

    slP.addEventListener('input', render);

    render();
  }

  function tryInitSimEP0410(){
    var root = document.getElementById('sim-ep0410-rotulacao');
    if (root) initSimEP0410(root); else setTimeout(tryInitSimEP0410, 200);
  }
  tryInitSimEP0410();
})();
</script>
</div>
""")

**Figura 4.10:** Simulateur EP04_10: Séparation des Blobs, Étiquetage et Descripteurs


<figure id="fig-04-sim-ep0410-rotulacao">
  <img src="imagens/fig-04-sim-ep0410-rotulacao.png" alt=" Simulateur EP04_10: Séparation des Blobs, Étiquetage et Descripteurs " style="max-width:80%" />
  <figcaption><strong>Figura 4.10:</strong>  Simulateur EP04_10: Séparation des Blobs, Étiquetage et Descripteurs </figcaption>
</figure>

In [ ]:
%%writefile EP04_10.py
# Code Python

In [ ]:
TestSuite("EP04_10.py").run()